# CheXpert frontal-subset TorchXRayVision inference

This Colab workflow performs deterministic extraction, QC, pretrained inference, evaluation, and metric-only plotting. It performs no training. Raw images, zip files, prediction tables with local paths, and the displayed sample grid must not be committed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configure paths
Edit these portable Drive-relative locations for your account.

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE_ROOT / 'datasets/CheXpert-v1.0-small.zip'
OUTPUT_ROOT = DRIVE_ROOT / 'medshiftlab/chexpert_small_frontal1000_torchxrayvision'
REPO_DIR = Path('/content/MedShiftLab-CXR')  # clone or upload the repository here
WORK_DIR = Path('/content/chexpert_work')
METADATA_DIR = WORK_DIR / 'metadata'
SMOKE_DIR = WORK_DIR / 'image_subsets/smoke20'
FULL_DIR = WORK_DIR / 'image_subsets/frontal1000'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.is_file(), f'Missing source archive: {ZIP_PATH}'
assert (REPO_DIR / 'scripts').is_dir(), f'Missing repository: {REPO_DIR}'
ZIP_PATH

## Extract metadata only
The image archive remains in Drive; this step extracts only `train.csv` and `valid.csv`.

In [ ]:
import zipfile
import pandas as pd
METADATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in ('CheXpert-v1.0-small/train.csv', 'CheXpert-v1.0-small/valid.csv'):
        archive.extract(member, METADATA_DIR)
train_csv = METADATA_DIR / 'CheXpert-v1.0-small/train.csv'
valid_csv = METADATA_DIR / 'CheXpert-v1.0-small/valid.csv'
train = pd.read_csv(train_csv)
valid = pd.read_csv(valid_csv)
print('train.csv shape:', train.shape)
print('valid.csv shape:', valid.shape)
print('frontal train rows:', int((train['Frontal/Lateral'] == 'Frontal').sum()))

## Smoke-20 extraction and QC

In [ ]:
import subprocess, sys
def run_script(name, *args):
    command = [sys.executable, str(REPO_DIR / 'scripts' / name), *map(str, args)]
    print(' '.join(command))
    subprocess.run(command, cwd=REPO_DIR, check=True)
run_script('extract_chexpert_image_subset.py', '--zip-path', ZIP_PATH, '--metadata-csv', train_csv, '--output-dir', SMOKE_DIR, '--limit', 20)
run_script('audit_chexpert_images.py', '--image-dir', SMOKE_DIR, '--output-json', OUTPUT_ROOT / 'qc_smoke20.json')

### Visual smoke check (display only)
The following grid is ephemeral QC. **Do not save or commit the grid or any source image.**

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
smoke_images = sorted(p for p in SMOKE_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
for axis, path in zip(axes.flat, smoke_images):
    with Image.open(path) as image:
        axis.imshow(image, cmap='gray')
    axis.axis('off')
fig.suptitle('Ephemeral smoke20 QC — DO NOT COMMIT')
plt.tight_layout(); plt.show(); plt.close(fig)

## Frontal-1000 extraction and QC
Both long loops show progress bars via `tqdm`.

In [ ]:
run_script('extract_chexpert_image_subset.py', '--zip-path', ZIP_PATH, '--metadata-csv', train_csv, '--output-dir', FULL_DIR, '--limit', 1000)
run_script('audit_chexpert_images.py', '--image-dir', FULL_DIR, '--output-json', OUTPUT_ROOT / 'qc_summary.json')

## Install and verify inference environment

In [ ]:
%pip install -q torchxrayvision==1.5.2 tqdm
import torch, torchxrayvision as xrv
assert torch.cuda.is_available(), 'Enable a GPU runtime before inference'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)
print('torchxrayvision:', xrv.__version__)
probe = xrv.models.DenseNet(weights='densenet121-res224-all')
print('pathologies:', probe.pathologies)
del probe

## Smoke-20 inference
Inference batches display a `tqdm` progress bar.

In [ ]:
run_script('run_torchxrayvision_inference.py', '--manifest-csv', SMOKE_DIR / 'subset_metadata.csv', '--image-root', SMOKE_DIR, '--output-csv', OUTPUT_ROOT / 'smoke20_predictions.csv', '--device', 'cuda', '--batch-size', 16)

## Frontal-1000 inference

In [ ]:
predictions_csv = OUTPUT_ROOT / 'frontal1000_predictions.csv'
run_script('run_torchxrayvision_inference.py', '--manifest-csv', FULL_DIR / 'subset_metadata.csv', '--image-root', FULL_DIR, '--output-csv', predictions_csv, '--device', 'cuda', '--batch-size', 16)

## Evaluate, plot, and save to Drive
The plots contain aggregate metrics only. The prediction CSV remains in Drive and must not be committed.

In [ ]:
RESULTS_DIR = OUTPUT_ROOT / 'results'
FIGURES_DIR = OUTPUT_ROOT / 'figures'
run_script('evaluate_chexpert_image_predictions.py', '--predictions-csv', predictions_csv, '--output-dir', RESULTS_DIR, '--threshold', 0.5, '--n-bins', 10)
run_script('plot_chexpert_image_inference_results.py', '--metrics-csv', RESULTS_DIR / 'evaluation_label_metrics.csv', '--output-dir', FIGURES_DIR)
print('Saved run outputs to:', OUTPUT_ROOT)